In [1]:
!pip install -q deepeval groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 892.2/892.2 kB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.7/129.7 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 2.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
huggingface-hub 1.33.0 requires click<9.0.0,>=8.4.2, but you have click 8.3.3 which is incompatible.


In [2]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

Enter your Groq API key: ··········


In [6]:
from deepeval.models import OpenAIModel
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, SingleTurnParams

groq_judge = OpenAIModel(
    model="openai/gpt-oss-safeguard-20b",
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",
    temperature=0,
)

correctness_metric = GEval(
    name="Answer Correctness",
    criteria="""
    Evaluate whether the answer is factually correct
    and complete based on the question and retrieved context.

    Penalize incorrect facts, contradictions, and
    missing essential information.
    """,
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.RETRIEVAL_CONTEXT,
    ],
    model=groq_judge,
    threshold=0.7,
)

print("Groq judge configured successfully!")

Groq judge configured successfully!


In [7]:
test_case = LLMTestCase(
    input="What is the capital of Nepal?",
    actual_output="Kathmandu is the capital of Nepal.",
    retrieval_context=[
        "Nepal is a country in South Asia. "
        "Its capital city is Kathmandu."
    ],
)

correctness_metric.measure(test_case)

print("Score:", correctness_metric.score)
print("Reason:", correctness_metric.reason)
print("Passed:", correctness_metric.is_successful())

Output()

Score: 1.0
Reason: The answer correctly states Kathmandu as Nepal's capital, matching the only essential detail in the question and the retrieval context. No factual errors or contradictions appear, and the response is complete for the asked query.
Passed: True


In [8]:
wrong_test_case = LLMTestCase(
    input="What is the capital of Nepal?",
    actual_output="Pokhara is the capital of Nepal.",
    retrieval_context=[
        "Nepal's capital city is Kathmandu."
    ],
)

correctness_metric.measure(wrong_test_case)

print("Score:", correctness_metric.score)
print("Reason:", correctness_metric.reason)
print("Passed:", correctness_metric.is_successful())

Output()

Score: 0.0
Reason: The response incorrectly states Pokhara as the capital, which contradicts the retrieval context that lists Kathmandu as the capital. This factual error violates steps 1-3 and shows a complete omission of the correct information, leading to a minimal score.
Passed: False


In [9]:
!pip install -q -U deepeval litellm pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.8/38.8 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.3/278.3 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 5.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires 

In [10]:
import os
import getpass

os.environ["GROQ_API_KEY"] = getpass.getpass(
    "Enter your Groq API key: "
)

print("API key configured.")

Enter your Groq API key: ··········
API key configured.


In [11]:
import pandas as pd

dataset = [
    {
        "id": "RAG-001",
        "question": "How long do I have to return an item?",
        "retrieval_context": [
            "Customers can return unused items within 30 days "
            "of delivery. Items must be in their original packaging."
        ],
        "golden_answer": (
            "You can return an unused item within 30 days of "
            "delivery. It must be in its original packaging."
        ),
        "rag_output": (
            "You can return an unused item within 30 days of "
            "delivery, provided it is in its original packaging."
        ),
    },
    {
        "id": "RAG-002",
        "question": "When will my refund arrive?",
        "retrieval_context": [
            "Approved refunds are processed within 5-7 business "
            "days after the returned item passes inspection."
        ],
        "golden_answer": (
            "After the returned item passes inspection, an "
            "approved refund is processed within 5-7 business days."
        ),
        "rag_output": (
            "Your refund will be processed within 5-7 business days."
        ),
    },
    {
        "id": "RAG-003",
        "question": "Can I cancel an order after it ships?",
        "retrieval_context": [
            "Orders can be cancelled before shipment. After "
            "shipment, cancellation is unavailable; customers "
            "may request a return after delivery."
        ],
        "golden_answer": (
            "You cannot cancel an order after it ships. After "
            "delivery, you may request a return."
        ),
        "rag_output": (
            "Yes, you can cancel your order after it ships by "
            "contacting customer support."
        ),
    },
    {
        "id": "RAG-004",
        "question": "Is standard shipping free?",
        "retrieval_context": [
            "Standard shipping is free for orders of NPR 2,000 "
            "or more. Orders below NPR 2,000 incur a shipping "
            "fee of NPR 150."
        ],
        "golden_answer": (
            "Standard shipping is free for orders of NPR 2,000 "
            "or more. Orders below NPR 2,000 have a NPR 150 fee."
        ),
        "rag_output": (
            "Standard shipping is free for every order, regardless "
            "of its value."
        ),
    },
    {
        "id": "RAG-005",
        "question": "How can I contact customer support?",
        "retrieval_context": [
            "Customer support is available by email at "
            "support@example.com, Monday to Friday, 9 AM to 5 PM."
        ],
        "golden_answer": (
            "Email support@example.com from Monday to Friday, "
            "9 AM to 5 PM."
        ),
        "rag_output": (
            "Yo, just shoot an email to support@example.com "
            "whenever. They're always there, 24/7!"
        ),
    },
]

df = pd.DataFrame(dataset)

df[["id", "question", "golden_answer", "rag_output"]]

,id,question,golden_answer,rag_output
0,RAG-001,How long do I have to return an item?,You can return an unused item within 30 days o...,You can return an unused item within 30 days o...
1,RAG-002,When will my refund arrive?,"After the returned item passes inspection, an ...",Your refund will be processed within 5-7 busin...
2,RAG-003,Can I cancel an order after it ships?,You cannot cancel an order after it ships. Aft...,"Yes, you can cancel your order after it ships ..."
3,RAG-004,Is standard shipping free?,"Standard shipping is free for orders of NPR 2,...","Standard shipping is free for every order, reg..."
4,RAG-005,How can I contact customer support?,Email support@example.com from Monday to Frida...,"Yo, just shoot an email to support@example.com..."


In [22]:
from deepeval.models import OpenAIModel
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, SingleTurnParams

# Using OpenAIModel wrapper over Groq API
judge = OpenAIModel(
    model="openai/gpt-oss-20b",
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",
    temperature=0,
)

# 1. Correctness: Is the answer factually right?
correctness = GEval(
    name="Correctness",
    criteria="""
    Evaluate factual agreement with the reference answer
    and retrieved context. Ignore stylistic differences.

    1.0: All important factual claims are correct.
    0.5: Partly correct, with a minor factual error.
    0.0: Contains a major contradiction or false claim.

    Do not treat a missing detail alone as a factual error
    if the answer does not contradict it.
    """,
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.EXPECTED_OUTPUT,
        SingleTurnParams.RETRIEVAL_CONTEXT,
    ],
    threshold=0.80,
    model=judge,
)

# 2. Completeness: Are all essential points included?
completeness = GEval(
    name="Completeness",
    criteria="""
    Evaluate whether the answer includes all essential
    information from the reference answer.

    1.0: All essential information is included.
    0.5: Some important details are missing.
    0.0: Most essential information is missing.

    Check conditions, exceptions, deadlines, and fees.
    Do not penalize concise answers that are complete.
    """,
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.EXPECTED_OUTPUT,
        SingleTurnParams.RETRIEVAL_CONTEXT,
    ],
    threshold=0.75,
    model=judge,
)

# 3. Style: Is the answer clear and professional?
style = GEval(
    name="Style",
    criteria="""
    Evaluate presentation only, not factual correctness.

    1.0: Clear, professional, grammatically correct,
    and appropriate for customer support.
    0.5: Understandable but noticeably informal or awkward.
    0.0: Extremely unclear or inappropriate.

    Judge tone, grammar, and readability only.
    Do not penalize factual errors under this metric.
    """,
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT,
    ],
    threshold=0.70,
    model=judge,
)

metrics = {
    "correctness": correctness,
    "completeness": completeness,
    "style": style,
}

print("Groq judge and all three GEval metrics successfully corrected!")

Groq judge and all three GEval metrics successfully corrected!


In [24]:
import time
from IPython.display import display
import pandas as pd

results = []

for row in dataset:
    print(f"\nEvaluating {row['id']}...")

    test_case = LLMTestCase(
        input=row["question"],
        actual_output=row["rag_output"],
        expected_output=row["golden_answer"],
        retrieval_context=row["retrieval_context"],
    )

    result = {
        "id": row["id"],
        "question": row["question"],
    }

    for metric_name, metric in metrics.items():
        try:
            metric.measure(test_case)

            score = float(metric.score)
            passed = metric.is_successful()

            result[f"{metric_name}_score"] = score
            result[f"{metric_name}_pass"] = passed
            result[f"{metric_name}_reason"] = metric.reason

            print(
                f"  {metric_name}: {score:.2f} "
                f"| {'PASS' if passed else 'FAIL'}"
            )

        except Exception as e:
            result[f"{metric_name}_score"] = None
            result[f"{metric_name}_pass"] = None
            result[f"{metric_name}_reason"] = (
                f"{type(e).__name__}: {str(e)[:250]}"
            )
            print(f"  {metric_name}: ERROR - {type(e).__name__}")

        # Allow time between calls to reduce rate-limit pressure.
        time.sleep(8)

    results.append(result)

results_df = pd.DataFrame(results)

score_columns = [
    "id",
    "correctness_score",
    "completeness_score",
    "style_score",
]

display(
    results_df[score_columns].round(3)
)

Output()


Evaluating RAG-001...


  correctness: 1.00 | PASS


Output()

  completeness: 0.10 | FAIL


...

Output()

  completeness: 0.05 | FAIL


Output()

  style: 0.05 | FAIL


,id,correctness_score,completeness_score,style_score
0,RAG-001,1.0,0.10,0.10
1,RAG-002,1.0,0.05,1.00
2,RAG-003,0.0,0.00,0.10
3,RAG-004,0.0,0.00,1.00
4,RAG-005,0.0,0.05,0.05


In [25]:
summary = []

for metric_name in metrics:
    score_col = f"{metric_name}_score"
    pass_col = f"{metric_name}_pass"

    valid_scores = results_df[score_col].dropna()
    valid_passes = results_df[pass_col].dropna()

    summary.append({
        "Metric": metric_name.title(),
        "Average Score": (
            round(valid_scores.mean(), 3)
            if not valid_scores.empty else None
        ),
        "Pass Rate (%)": (
            round(valid_passes.astype(bool).mean() * 100, 1)
            if not valid_passes.empty else None
        ),
        "Evaluated": len(valid_scores),
        "Failed / Errors": len(dataset) - len(valid_scores),
    })

summary_df = pd.DataFrame(summary)
display(summary_df)

,Metric,Average Score,Pass Rate (%),Evaluated,Failed / Errors
0,Correctness,0.40,40.0,5,0
1,Completeness,0.04,0.0,5,0
2,Style,0.45,40.0,5,0


In [23]:
import time
import pandas as pd
from IPython.display import display

results = []

for row in dataset:
    test_case = LLMTestCase(
        input=row["question"],
        actual_output=row["rag_output"],
        expected_output=row["golden_answer"],
        retrieval_context=row["retrieval_context"],
    )

    result = {
        "Test Case": row["id"],
        "Question": row["question"],
    }

    for name, metric in metrics.items():
        try:
            metric.measure(test_case)
            passed = metric.is_successful()

            result[f"{name}"] = "✅ PASS" if passed else "❌ FAIL"
            result[f"{name} Score"] = round(float(metric.score), 2)
            result[f"{name} Reason"] = (
                "—" if passed else (metric.reason or "Criteria not met")
            )

        except Exception as e:
            result[name] = "⚠️ ERROR"
            result[f"{name} Score"] = None
            result[f"{name} Reason"] = f"{type(e).__name__}: {str(e)[:200]}"

        time.sleep(2)

    statuses = [result[name] for name in metrics]
    result["Overall"] = (
        "⚠️ ERROR" if "⚠️ ERROR" in statuses
        else "❌ FAIL" if "❌ FAIL" in statuses
        else "✅ PASS"
    )

    results.append(result)

results_df = pd.DataFrame(results)

# Compact test-case overview
overview_cols = ["Test Case", "Overall"] + list(metrics.keys())
display(results_df[overview_cols].style.hide(axis="index"))

# Show only failed tests, with reasons
failure_rows = []

for result in results:
    for name in metrics:
        if result[name] in ("❌ FAIL", "⚠️ ERROR"):
            failure_rows.append({
                "Test Case": result["Test Case"],
                "Question": result["Question"],
                "Failed Metric": name,
                "Score": result[f"{name} Score"],
                "Why It Failed": result[f"{name} Reason"],
            })

print("\n🔍 FAILURE ANALYSIS")

if failure_rows:
    display(pd.DataFrame(failure_rows).style.hide(axis="index"))
else:
    print("✅ All test cases passed.")

# Save the full report
results_df.to_csv("rag_test_report.csv", index=False)
print("\n📄 Full report saved: rag_test_report.csv")

Output()

Output()

Output()

Output()

Output()

...

Output()

Test Case,Overall,correctness,completeness,style
RAG-001,❌ FAIL,✅ PASS,❌ FAIL,❌ FAIL
RAG-002,❌ FAIL,❌ FAIL,❌ FAIL,✅ PASS
RAG-003,❌ FAIL,❌ FAIL,❌ FAIL,❌ FAIL
RAG-004,❌ FAIL,❌ FAIL,❌ FAIL,✅ PASS
RAG-005,⚠️ ERROR,❌ FAIL,⚠️ ERROR,❌ FAIL



🔍 FAILURE ANALYSIS


Test Case,Question,Failed Metric,Score,Why It Failed
RAG-001,How long do I have to return an item?,completeness,0.100000,The actual output includes all essential details from the expected answer—30‑day return window and requirement for original packaging—matching the retrieval context and meeting the evaluation criteria for completeness and conciseness.
RAG-001,How long do I have to return an item?,style,0.100000,"The response maintains a professional, customer‑support tone, is grammatically correct and easy to read, and directly addresses the question. It meets all evaluation criteria for a clear, professional reply, earning the maximum score."
RAG-002,When will my refund arrive?,correctness,0.100000,"The response correctly states that refunds are processed within 5‑7 business days, matching the factual claim in the retrieval context. It omits the additional condition about inspection, but this omission does not contradict the reference answer, so no penalty is applied."
RAG-002,When will my refund arrive?,completeness,0.050000,"The answer correctly states the 5‑7 business day deadline but omits the essential condition that the refund is processed only after the returned item passes inspection, which is present in both the expected answer and the retrieval context. This missing key detail reduces the score to 0.5."
RAG-003,Can I cancel an order after it ships?,correctness,0.000000,"The actual output states that an order can be cancelled after it ships, which directly contradicts both the expected answer and the retrieval context that say cancellation is unavailable after shipment. This is a clear contradiction, so the score is 0."
RAG-003,Can I cancel an order after it ships?,completeness,0.000000,"The actual answer incorrectly states that cancellation is possible after shipment, contradicting the expected answer and retrieval context, which both state cancellation is unavailable. Key details such as the condition that cancellation cannot be done post-shipment and the possibility of a return are missing, leading to a 0.0 score."
RAG-003,Can I cancel an order after it ships?,style,0.100000,"The response maintains a professional, customer‑support tone, is grammatically correct, and is clear and concise, meeting the criteria for a 1.0 score."
RAG-004,Is standard shipping free?,correctness,0.000000,"The actual output states that standard shipping is free for every order, which contradicts both the expected answer and the retrieval context that specify free shipping only for orders of NPR 2,000 or more and a fee for smaller orders. This direct contradiction warrants a score of 0."
RAG-004,Is standard shipping free?,completeness,0.000000,"The actual answer omits the key threshold of NPR 2,000 and the NPR 150 fee for orders below that amount, which are essential details present in both the expected answer and the retrieval context. Because these critical pieces of information are missing, the response fails to meet the evaluation criteria and receives a score of 0."
RAG-005,How can I contact customer support?,correctness,0.000000,"The actual output claims 24/7 availability, which contradicts the expected answer and the retrieval context that specify support is only available Monday to Friday, 9 AM to 5 PM."



📄 Full report saved: rag_test_report.csv
